# LionAG2: Recursive Exploratory Research with AG2 beta — events and reactive orchestration (4/10)

In [Day 3](03_multi_agent_handoff.ipynb) we wired a surveyor to theorists by hand — extract questions, build prompts, fan out with `asyncio.gather`. That works, but it means every coordination step is imperative code you write and maintain.

AG2's **event system** offers a different angle. Every `agent.ask()` call produces a stream of **typed events** — not log strings, but Python objects with real attributes like `ToolCallEvent.name` and `ModelResponse.usage`. In this tutorial we'll inspect what those events look like, then see how they carry through to sub-agent calls.

In [1]:
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from autogen.beta import Agent, MemoryStream
from autogen.beta.config import OpenAIConfig
from autogen.beta.tools import ExaToolkit

load_dotenv()

config = OpenAIConfig(
    model="gpt-5.4-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
)
exa = ExaToolkit(api_key=os.getenv("EXA_API_KEY"))

## Walking the event stream

When you pass a `MemoryStream` to `agent.ask()`, every event the agent produces is recorded. Let's run a single agent with a tool and see what comes back.

In [2]:
researcher = Agent("researcher", config=config, tools=[exa])
stream = MemoryStream()

reply = await researcher.ask(
    "Find one recent paper on room-temperature superconductivity. "
    "Give the title and one key finding.",
    stream=stream,
)

print(reply.body)

One recent paper is **“Room-Temperature Superconductivity at 298 K in Ternary La-Sc-H System at High-pressure Conditions.”**

**Key finding:** The authors report superconductivity with an onset temperature up to **298 K** in a **LaSc₂H₂₄** hydride synthesized under **very high pressure** (about **250–260 GPa**), supported by zero-resistance and magnetic-field-suppression measurements.

If you want, I can also give you a more conservative, widely accepted recent paper on superconductivity that’s closer to ambient pressure.


In [3]:
for ev in await stream.history.get_events():
    print(f"  {type(ev).__name__}: {str(ev)[:90]}")

  ModelRequest: ModelRequest(parts=[TextInput(content='Find one recent paper on room-temperature supercond
  ModelResponse: ModelResponse(content=None, tool_calls=ToolCallsEvent(calls=[ToolCallEvent(id=call_lhzgGMV
  ToolCallsEvent: ToolCallsEvent(calls=[ToolCallEvent(id=call_lhzgGMVZXNzaA8pU14zw36fh, name='exa_search', a
  ToolCallEvent: ToolCallEvent(id=call_lhzgGMVZXNzaA8pU14zw36fh, name='exa_search', arguments='{"query":"re
  ToolResultEvent: ToolResultEvent(parent_id=call_lhzgGMVZXNzaA8pU14zw36fh, name='exa_search', result=ToolRes
  ToolResultsEvent: ToolResultsEvent(results=[ToolResultEvent(parent_id=call_lhzgGMVZXNzaA8pU14zw36fh, name='e
  ModelResponse: ModelResponse(content=One recent paper is **“Room-Temperature Superconductivity at 298 K i


A single tool-using turn produced this sequence:

1. **ModelRequest** — the user prompt bundled with tool definitions, sent to the LLM.
2. **ModelResponse** (no content) — the model's first reply is a tool call, not text.
3. **ToolCallEvent** — AG2 parsed the model's tool-call request into a function invocation with name and arguments.
4. **ToolResultsEvent** — the tool ran and returned results, which get fed back into the model's context.
5. **ModelResponse** (with content) — the model processes everything and produces the final answer.

These are typed Python objects, not strings. `ToolCallEvent.name`, `ModelResponse.usage.total_tokens` — real attributes you can filter on, react to, or aggregate. That's the foundation for everything that follows.

## Sub-agents produce events too

In Day 3 we dispatched theorists manually with `asyncio.gather`. AG2 beta has a shortcut: `agent.as_tool()` wraps one agent as a callable tool for another. The parent agent decides when to call it, just like any other tool.

The key insight: **each sub-agent emits the same kinds of events on its own stream**. By passing separate `MemoryStream` objects, we can inspect what each agent did independently — useful for debugging, cost tracking, or understanding why a multi-agent run produced what it did.

In [4]:
topic = "high-Tc superconductivity"


class Findings(BaseModel):
    claim: str = Field(description="The main claim of the research finding.")
    novelty: float = Field(description="How novel the finding is, 0.0 to 1.0.")
    evidence: str = Field(description="Summary of evidence supporting the claim.")
    open_questions: list[str] = Field(description="Unresolved questions or uncertainties.")


surveyor_stream = MemoryStream()
critic_stream = MemoryStream()
research_stream = MemoryStream()

surveyor = Agent(
    "surveyor",
    config=config,
    tools=[exa],
    prompt=(
        f"You are a literature surveyor specializing in {topic}. "
        "Search broadly, then summarize the landscape of current research."
    ),
)

critic = Agent(
    "critic",
    config=config,
    prompt=(
        "You are a skeptical reviewer. Challenge assumptions, identify weak "
        "evidence, and poke holes in hypothesis arguments. Be specific."
    ),
)

researcher = Agent(
    "researcher",
    config=config,
    tools=[
        exa,
        surveyor.as_tool(
            description="Survey literature on a topic",
            stream=surveyor_stream,
        ),
        critic.as_tool(
            description="Critically evaluate a hypothesis or claim",
            stream=critic_stream,
        ),
    ],
    prompt=(
        f"You are a research lead investigating {topic}. Your workflow: "
        "1) Ask the surveyor to gather recent literature. "
        "2) Form a hypothesis based on what the surveyor finds. "
        "3) Ask the critic to evaluate your hypothesis. "
        "4) Refine and produce your final finding."
    ),
    response_schema=Findings,
)

In [5]:
reply = await researcher.ask(
    f"Investigate {topic}: what is the most promising recent direction "
    "and what are the key open questions? Use the surveyor to gather "
    "literature, then form a hypothesis and have the critic evaluate it.",
    stream=research_stream,
)

finding = await reply.content(retries=2)
display(Markdown(
    f"**Claim:** {finding.claim}\n\n"
    f"**Novelty:** {finding.novelty:.2f}\n\n"
    f"**Evidence:** {finding.evidence}\n\n"
    f"**Open questions:**\n" + "\n".join(f"- {q}" for q in finding.open_questions)
))

**Claim:** The most promising recent direction in high-Tc superconductivity is the rapidly advancing nickelate platform—especially pressurized bilayer/trilayer Ruddlesden–Popper nickelates (La3Ni2O7, La4Ni3O10) and strain-stabilized ambient-pressure La3Ni2O7 films—because it is the newest strongly correlated oxide family to reach cuprate-scale Tc while becoming experimentally tractable enough to attack the pairing mechanism. Hydrides remain the strongest route to room-temperature Tc, but they are less informative for ambient-pressure correlated-electron physics and are still constrained by megabar pressures.

**Novelty:** 0.66

**Evidence:** Recent reviews and primary papers point to nickelates as the most active and conceptually promising recent oxide direction. Annual Review and Nature Reviews-style summaries emphasize that infinite-layer nickelates and, more recently, Ruddlesden–Popper nickelates now show superconductivity, transport anomalies, and multiple competing structural/electronic scenarios. In particular, La3Ni2O7 under pressure has Tc near 80 K, and 2024-2025 reports extended the phase landscape to ambient-pressure thin films and related RP compounds. These systems are attractive because they are the first non-cuprate, strongly correlated oxide family to show cuprate-like high Tc and because multiple probes now exist to interrogate the relevant questions: bilayer exchange, orbital selectivity, the gamma/dz2 pocket, oxygen stoichiometry, structural polymorphism, and bulk-vs-filamentary superconductivity. 

By contrast, recent hydride reviews conclude that hydride superconductivity is real and remains the clearest route toward very high or room-temperature Tc, but the field is still dominated by megabar-pressure synthesis, reproducibility concerns, and measurement challenges. Hydrides are therefore more promising for maximal Tc than for learning a transferable ambient-pressure mechanism. Cuprates remain central as the mechanism benchmark, especially through charge-order and pseudogap work, but the recent literature is more about deepening understanding than opening a new materials frontier. 

So the best current synthesis is: if the goal is 'most promising recent direction' in a broad high-Tc sense, hydrides lead on record Tc; if the goal is the most promising recent direction for new physics and a potentially transferable ambient-pressure route in correlated oxides, bilayer/trilayer nickelates are the leading candidate.

**Open questions:**
- Is superconductivity in La3Ni2O7 and related RP nickelates truly bulk and intrinsic, or partly filamentary / phase-mixture-driven?
- What is the correct pairing symmetry in RP nickelates: d-wave, s±, extended s-wave, or a more mixed state?
- Is the dz2/gamma pocket essential to pairing, and how does its presence change with pressure, strain, and oxygen stoichiometry?
- What is the dominant pairing glue: interlayer antiferromagnetic exchange, in-plane spin fluctuations, orbital-selective correlations, or a multiband mechanism?
- How do structural polymorphism, oxygen ordering, and epitaxial strain control the superconducting phase boundaries in nickelates?
- Can the nickelate mechanism be generalized to other RP or hybrid nickelates, and can it be stabilized robustly at ambient pressure?
- For hydrides, can high Tc be retained while reducing pressure and improving direct Meissner/phase-diagram evidence?
- For cuprates, can charge order, pseudogap physics, and pairing mechanism be unified into a predictive theory that informs materials design?

Now let's inspect what each agent did. Each stream captured the events for one agent independently:

In [6]:
named_streams = [
    ("researcher", research_stream),
    ("surveyor", surveyor_stream),
    ("critic", critic_stream),
]

for agent_name, s in named_streams:
    events = await s.history.get_events()
    print(f"\n{'='*60}")
    print(f"  {agent_name} — {len(events)} events")
    print(f"{'='*60}")
    for ev in events:
        print(f"  {type(ev).__name__}: {str(ev)[:90]}")


  researcher — 21 events
  ModelRequest: ModelRequest(parts=[TextInput(content='Investigate high-Tc superconductivity: what is the 
  ModelResponse: ModelResponse(content=None, tool_calls=ToolCallsEvent(calls=[ToolCallEvent(id=call_N3kZMXT
  ToolCallsEvent: ToolCallsEvent(calls=[ToolCallEvent(id=call_N3kZMXTyRHIIq8h5CP7jRBfb, name='task_surveyor'
  ToolCallEvent: ToolCallEvent(id=call_N3kZMXTyRHIIq8h5CP7jRBfb, name='task_surveyor', arguments='{"objecti
  ToolErrorEvent: ToolErrorEvent(parent_id=call_N3kZMXTyRHIIq8h5CP7jRBfb, name='task_surveyor', result=ToolR
  ToolResultsEvent: ToolResultsEvent(results=[ToolErrorEvent(parent_id=call_N3kZMXTyRHIIq8h5CP7jRBfb, name='ta
  ModelResponse: ModelResponse(content=None, tool_calls=ToolCallsEvent(calls=[ToolCallEvent(id=call_5ChUGiQ
  ToolCallsEvent: ToolCallsEvent(calls=[ToolCallEvent(id=call_5ChUGiQInz1Le0tbipZMOkZ7, name='exa_search', a
  ToolCallEvent: ToolCallEvent(id=call_5ChUGiQInz1Le0tbipZMOkZ7, name='exa_search', arguments='{"query": 

The researcher's stream shows the full orchestration — ModelRequest, tool calls to the surveyor and critic (which appear as `ToolCallEvent`s), their results coming back, and the final ModelResponse. The surveyor and critic streams show their own internal work: search calls, model reasoning, tool results.

Same event types at every level. That uniformity is what makes the system debuggable — you don't need different tools to understand what happened at each layer of a multi-agent run.

## Up next

Day 5: **reactive observers**.

So far we inspected events after the fact. But what if you could react to events *as they happen* — count tool calls, track token usage, or dispatch new agents the moment a result comes in? That's what observers do.

- [Day 1: Simple search agent with Exa](01_get_started.ipynb)
- [Day 2: Structured output with response_schema](02_typed_findings.ipynb)
- [Day 3: Typed multi-agent handoff](03_multi_agent_handoff.ipynb)
- [AG2 beta docs](https://docs.ag2.ai/)